# Pairs Trading with Wavelet Transform
**Réplication — Eroğlu, Yener & Yiğit (2023), *Quantitative Finance* 23:7-8**

Ce notebook exécute et compare les **4 méthodes** du papier sur les 7 périodes :

| Méthode | Paires | Prix |
|---|---|---|
| MD Standard | Distance minimale | Bruts |
| MD Wavelet | Distance minimale | Filtrés sym22 |
| CI Standard | Coïntégration (Johansen) | Bruts |
| CI Wavelet | Coïntégration (Johansen) | Filtrés sym22 |


## 0. Imports

In [ ]:
import sys, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
from scipy import stats

warnings.filterwarnings("ignore")
sys.path.insert(0, ".")

from wavelet_filter import filter_price_matrix
from pair_selection import minimum_distance_pairs, cointegration_pairs
from spread_estimation import estimate_all_pairs, build_spread
from trading_engine import run_trading_period
from performance import compute_period_metrics, spread_stationarity_rate

# ── Plot style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "legend.fontsize": 9,
})
COLORS = {
    "md_standard":    "#2c7bb6",
    "md_wavelet":     "#d7191c",
    "coint_standard": "#1a9641",
    "coint_wavelet":  "#fdae61",
}
LABELS = {
    "md_standard":    "MD Standard",
    "md_wavelet":     "MD Wavelet (sym22)",
    "coint_standard": "CI Standard",
    "coint_wavelet":  "CI Wavelet (sym22)",
}
LINESTYLES = {
    "md_standard": "--", "md_wavelet": "-",
    "coint_standard": ":", "coint_wavelet": "-",
}
print("Modules chargés ✓")

## 1. Chargement des données

Modifier `DATA_PATH` avec le chemin de votre fichier CSV de prix ajustés SP500.


In [ ]:
DATA_PATH = "SP100_tickers_adj_close.csv"

try:
    prices = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True).sort_index()
    prices = prices.dropna(axis=1, how="any")
    SYNTHETIC = False
    print(f"Chargé : {prices.shape[0]} jours × {prices.shape[1]} tickers")
    print(f"Période : {prices.index[0].date()} → {prices.index[-1].date()}")
except FileNotFoundError:
    print(f"'{DATA_PATH}' introuvable — données synthétiques (50 stocks, 1512 jours)")
    np.random.seed(42); T, N = 1512, 50
    dates = pd.bdate_range("2010-03-05", periods=T)
    tickers = [f"STK{i:03d}" for i in range(N)]
    common = np.cumsum(np.random.randn(T)*0.01)
    returns = common[:,None]*0.5 + np.random.randn(T,N)*0.015
    prices = pd.DataFrame(100*np.exp(np.cumsum(returns,axis=0)), index=dates, columns=tickers)
    SYNTHETIC = True
    print(f"Synthétique : {prices.shape[0]} × {prices.shape[1]}")
prices.head(3)

## 2. Découpage train / trade (Table 1 du papier)

In [ ]:
PERIOD_LENGTH = 252
N_PERIODS     = 7

periods = []
for i in range(N_PERIODS):
    t0, t1, t2 = i*PERIOD_LENGTH, (i+1)*PERIOD_LENGTH, (i+2)*PERIOD_LENGTH
    if t2 > len(prices): t2 = len(prices)
    train_p, trade_p = prices.iloc[t0:t1], prices.iloc[t1:t2]
    if len(train_p) >= 50 and len(trade_p) >= 20:
        periods.append((train_p, trade_p))

pd.DataFrame([{
    "Période": i+1,
    "Train début": p[0].index[0].date(), "Train fin": p[0].index[-1].date(),
    "Trade début": p[1].index[0].date(), "Trade fin":  p[1].index[-1].date(),
    "N tickers":  p[0].shape[1],
} for i,p in enumerate(periods)])

## 3. Configuration du backtest

- `RUN_COINT = True` pour activer la coïntégration (lent : ~20-30 min pour 415 tickers)  
- `RUN_COINT = False` pour MD uniquement (rapide)


In [ ]:
WAVELET    = "sym22"
N_MD_PAIRS = 1000        # papier : 1000
THETA      = 0.001       # 10 bps par action
RUN_COINT  = False       # ← mettre True pour les 4 méthodes

print(f"Wavelet   : {WAVELET}")
print(f"Paires MD : {N_MD_PAIRS}")
print(f"Coût      : {THETA*100:.1f} bps")
print(f"Coint     : {'OUI' if RUN_COINT else 'NON (MD uniquement)'}")

## 4. Backtest complet — 7 périodes

In [ ]:
all_results = {k: [] for k in ["md_standard","md_wavelet","coint_standard","coint_wavelet"]}
stationarity = {k: [] for k in all_results}

for i, (train_p, trade_p) in enumerate(periods):
    print(f"\n── Période {i+1}/{len(periods)}: "
          f"{train_p.index[0].date()} → {trade_p.index[-1].date()} ──")

    common_t = list(set(train_p.columns) & set(trade_p.columns))
    train_p, trade_p = train_p[common_t], trade_p[common_t]

    train_f = filter_price_matrix(train_p, wavelet=WAVELET)
    trade_f = filter_price_matrix(trade_p, wavelet=WAVELET)

    # ── Minimum Distance ────────────────────────────────────────────────────
    md_p = minimum_distance_pairs(train_p, n_pairs=N_MD_PAIRS)
    for key, wav in [("md_standard", False), ("md_wavelet", True)]:
        params = estimate_all_pairs(md_p, train_p, train_f, wavelet=wav)
        res    = run_trading_period(params, trade_p, trade_f, wavelet=wav, theta=THETA)
        m      = compute_period_metrics(res, len(md_p))
        all_results[key].append(m)
        stationarity[key].append(spread_stationarity_rate(params, trade_p, trade_f, wavelet=wav))
        print(f"  MD-{'wav' if wav else 'std'}: ret={m['mean_return']:+.2%}  "
              f"Sharpe={m['sharpe_ratio']:.3f}  stat={stationarity[key][-1]:.1%}")

    # ── Coïntégration ───────────────────────────────────────────────────────
    if RUN_COINT:
        coint_p = cointegration_pairs(train_p, verbose=False)
        print(f"  Johansen : {len(coint_p)} paires coïntégrées")
        if len(coint_p) > 0:
            for key, wav in [("coint_standard", False), ("coint_wavelet", True)]:
                params = estimate_all_pairs(coint_p, train_p, train_f, wavelet=wav)
                res    = run_trading_period(params, trade_p, trade_f, wavelet=wav, theta=THETA)
                m      = compute_period_metrics(res, len(coint_p))
                all_results[key].append(m)
                stationarity[key].append(spread_stationarity_rate(params, trade_p, trade_f, wavelet=wav))
                print(f"  CI-{'wav' if wav else 'std'}: ret={m['mean_return']:+.2%}  "
                      f"Sharpe={m['sharpe_ratio']:.3f}  stat={stationarity[key][-1]:.1%}")

# Sauvegarder
with open("results.pkl", "wb") as f:
    pickle.dump({"results": all_results, "stationarity": stationarity, "periods": periods}, f)
print("\n✓ Backtest terminé — sauvegardé dans results.pkl")

## 5. Statistiques de base — Table 4 du papier

Rendements, écart-type, skewness, kurtosis sur les 7 périodes.


In [ ]:
def agg_table(key, label):
    ms = [m for m in all_results[key] if m]
    if not ms: return pd.DataFrame()
    rows = []
    for stat, fmt in [
        ("mean_return",  "{:+.2%}"),
        ("std_return",   "{:.4f}"),
        ("skewness",     "{:+.4f}"),
        ("kurtosis",     "{:.1f}"),
        ("sharpe_ratio", "{:.4f}"),
        ("max_drawdown", "{:.2%}"),
        ("pct_positive", "{:.1f}%"),
        ("var_5pct",     "{:.2%}"),
        ("cvar_5pct",    "{:.2%}"),
    ]:
        vals = [m[stat] for m in ms]
        rows.append({
            "Statistique": stat,
            f"{label} Min":  fmt.format(min(vals)),
            f"{label} Max":  fmt.format(max(vals)),
            f"{label} Mean": fmt.format(np.mean(vals)),
        })
    return pd.DataFrame(rows).set_index("Statistique")

# Construire les tables disponibles
tables = {}
for key in all_results:
    if all_results[key]:
        tables[key] = agg_table(key, LABELS[key])

# Afficher les méthodes disponibles côte à côte
available = [k for k in ["md_standard","md_wavelet","coint_standard","coint_wavelet"]
             if k in tables]
if available:
    result = tables[available[0]]
    for k in available[1:]:
        result = result.join(tables[k])
    result

## 6. Rendements cumulés — Figure 4 du papier

Comparaison des 4 méthodes (ou 2 si coïntégration désactivée) sur les 7 périodes.


In [ ]:
available = [k for k in ["md_standard","md_wavelet","coint_standard","coint_wavelet"]
             if all_results[k]]

n_periods = len(periods)
ncols = 4
nrows = (n_periods + ncols - 1) // ncols  # ceil division

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4*nrows))
axes = axes.flatten()

for i, (train_p, trade_p) in enumerate(periods):
    ax = axes[i]
    for key in available:
        if i >= len(all_results[key]) or not all_results[key][i]:
            continue
        cr = all_results[key][i]["cum_returns"] * 100
        ax.plot(trade_p.index[:len(cr)], cr,
                label=LABELS[key], color=COLORS[key],
                linestyle=LINESTYLES[key], linewidth=1.4)
    ax.axhline(0, color="black", linewidth=0.6, alpha=0.4)
    ax.set_title(f"Période {i+1}  ({trade_p.index[0].strftime('%b %Y')})", fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    if i == 0:
        ax.legend(fontsize=8, ncol=1)

for j in range(n_periods, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Rendements cumulés par période — 4 méthodes", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("fig4_cumulative_returns.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sauvegardé : fig4_cumulative_returns.png")

## 7. Sharpe ratios par période — Table 5 / Figure 5 du papier

In [ ]:
available = [k for k in ["md_standard","md_wavelet","coint_standard","coint_wavelet"]
             if all_results[k]]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# ── Gauche : Sharpe par période ─────────────────────────────────────────────
x = np.arange(1, len(periods)+1)
width = 0.8 / len(available)

for idx, key in enumerate(available):
    ms = [m for m in all_results[key] if m]
    sharpes = [m["sharpe_ratio"] for m in ms]
    if not sharpes: continue
    offset = (idx - len(available)/2 + 0.5) * width
    ax1.bar(x[:len(sharpes)] + offset, sharpes, width=width,
            label=LABELS[key], color=COLORS[key], alpha=0.85)

ax1.axhline(0, color="black", linewidth=0.8)
ax1.set_xlabel("Période"); ax1.set_ylabel("Sharpe ratio (annualisé)")
ax1.set_title("Sharpe ratio par période")
ax1.set_xticks(x[:len(periods)])
ax1.legend(fontsize=8)

# ── Droite : Moyennes et IC ─────────────────────────────────────────────────
labels_plot, means, errors = [], [], []
for key in available:
    ms = [m for m in all_results[key] if m]
    if not ms: continue
    vals = [m["sharpe_ratio"] for m in ms]
    labels_plot.append(LABELS[key])
    means.append(np.mean(vals))
    errors.append(np.std(vals, ddof=1) / np.sqrt(len(vals)) * 1.96)  # IC 95%

y_pos = np.arange(len(labels_plot))
bars = ax2.barh(y_pos, means, xerr=errors, capsize=5,
                color=[COLORS[k] for k in available if all_results[k]],
                alpha=0.85, height=0.5)
ax2.axvline(0, color="black", linewidth=0.8)
ax2.set_yticks(y_pos); ax2.set_yticklabels(labels_plot)
ax2.set_xlabel("Sharpe moyen (± IC 95%)")
ax2.set_title("Sharpe moyen sur 7 périodes")

plt.tight_layout()
plt.savefig("fig5_sharpe_ratios.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Standard vs Wavelet — comparaison directe

Amélioration apportée par le filtre wavelet pour chaque méthode de sélection.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

metrics = [
    ("mean_return",  "Rendement moyen",  "{:+.2%}"),
    ("sharpe_ratio", "Sharpe ratio",      "{:.2f}"),
    ("max_drawdown", "Max Drawdown",      "{:.2%}"),
    ("pct_positive", "% Trades positifs", "{:.1f}%"),
    ("var_5pct",     "VaR (5%)",          "{:.2%}"),
    ("cvar_5pct",    "CVaR (5%)",         "{:.2%}"),
]

x = np.arange(1, len(periods)+1)

for ax, (stat, title, fmt) in zip(axes.flatten(), metrics):
    for key in ["md_standard","md_wavelet","coint_standard","coint_wavelet"]:
        ms = [m for m in all_results[key] if m]
        if not ms: continue
        vals = [m[stat] for m in ms]
        ax.plot(x[:len(vals)], vals,
                label=LABELS[key], color=COLORS[key],
                linestyle=LINESTYLES[key], linewidth=1.5,
                marker="o", markersize=4)
    ax.axhline(0, color="black", linewidth=0.5, alpha=0.3)
    ax.set_title(title); ax.set_xlabel("Période")
    ax.set_xticks(x[:len(periods)])
    if ax == axes.flatten()[0]:
        ax.legend(fontsize=8, ncol=2)

plt.suptitle("Comparaison des 4 méthodes — 7 périodes", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("fig_comparison_all.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Stationnarité des spreads — Table 8 du papier

Fraction des spreads rejetant la racine unitaire (ADF 5%) pendant la période de trading.


In [ ]:
available = [k for k in ["md_standard","md_wavelet","coint_standard","coint_wavelet"]
             if stationarity[k]]

# Table
stat_data = {"Période": list(range(1, len(periods)+1))}
for key in available:
    if stationarity[key]:
        stat_data[LABELS[key]] = [f"{v:.1%}" for v in stationarity[key]]
stat_df = pd.DataFrame(stat_data).set_index("Période")

# Moyennes
means_row = {"Période": "Moyenne"}
for key in available:
    if stationarity[key]:
        means_row[LABELS[key]] = f"{np.mean(stationarity[key]):.1%}"
stat_df.loc["Moyenne"] = [means_row.get(LABELS[k], "") for k in available]

# Visualisation
fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(1, len(periods)+1)
for key in available:
    vals = stationarity[key]
    if vals:
        ax.plot(x[:len(vals)], [v*100 for v in vals],
                label=LABELS[key], color=COLORS[key],
                linestyle=LINESTYLES[key], linewidth=1.5, marker="o", markersize=5)
ax.set_xlabel("Période"); ax.set_ylabel("% spreads stationnaires (ADF 5%)")
ax.set_title("Table 8 — Taux de rejet de racine unitaire (spreads trading)")
ax.set_xticks(x[:len(periods)]); ax.legend()
plt.tight_layout()
plt.savefig("fig_stationarity.png", dpi=150, bbox_inches="tight")
plt.show()
print()
display(stat_df)

## 10. Tableau récapitulatif — Min / Max / Mean sur 7 périodes

In [ ]:
available = [k for k in ["md_standard","md_wavelet","coint_standard","coint_wavelet"]
             if all_results[k]]

rows = []
for key in available:
    ms = [m for m in all_results[key] if m]
    if not ms: continue
    def s(stat): return [m[stat] for m in ms]
    rows.append({
        "Méthode":         LABELS[key],
        "Ret. moyen":      f"{np.mean(s('mean_return')):+.2%}",
        "Ret. min":        f"{min(s('mean_return')):+.2%}",
        "Ret. max":        f"{max(s('mean_return')):+.2%}",
        "Sharpe moyen":    f"{np.mean(s('sharpe_ratio')):.3f}",
        "Max DD moyen":    f"{np.mean(s('max_drawdown')):.2%}",
        "% Positifs":      f"{np.mean(s('pct_positive')):.1f}%",
        "CVaR (5%) moy.":  f"{np.mean(s('cvar_5pct')):.2%}",
        "Stat. spreads":   f"{np.mean(stationarity[key]):.1%}" if stationarity[key] else "—",
    })

pd.DataFrame(rows).set_index("Méthode")

## 11. Visualisation d'un spread — Période 1

Illustration Figure 2 du papier : spread standard vs wavelet pour la meilleure paire MD.


In [ ]:
# Recalcul sur période 1 pour visualisation
train0, trade0 = periods[0]
common_t = list(set(train0.columns) & set(trade0.columns))
train0, trade0 = train0[common_t], trade0[common_t]

train_f0 = filter_price_matrix(train0, wavelet=WAVELET)
trade_f0 = filter_price_matrix(trade0, wavelet=WAVELET)

md_p0 = minimum_distance_pairs(train0, n_pairs=5)
si, sj = md_p0.iloc[0]["stock_i"], md_p0.iloc[0]["stock_j"]

params_std0 = estimate_all_pairs(md_p0, train0, wavelet=False)
params_wav0 = estimate_all_pairs(md_p0, train0, train_f0, wavelet=True)

p_s = params_std0[(si, sj)]
p_w = params_wav0[(si, sj)]

sp_std = build_spread(trade0[si].values, trade0[sj].values, p_s.alpha, p_s.beta)
sp_wav = build_spread(trade_f0[si].values, trade_f0[sj].values, p_w.alpha, p_w.beta)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, sp, p, label, color in [
    (axes[0], sp_std, p_s, "Standard", COLORS["md_standard"]),
    (axes[1], sp_wav, p_w, f"Wavelet ({WAVELET})", COLORS["md_wavelet"]),
]:
    ax.plot(trade0.index, sp, color=color, linewidth=0.9)
    ax.axhline(+p.threshold, color=color, linestyle="--", alpha=0.7,
               label=f"+2σ = {p.threshold:.3f}")
    ax.axhline(-p.threshold, color=color, linestyle="--", alpha=0.7,
               label=f"-2σ = {-p.threshold:.3f}")
    ax.axhline(0, color="black", linewidth=0.5)
    ax.fill_between(trade0.index, sp,  p.threshold, where=sp >  p.threshold, alpha=0.15, color=color)
    ax.fill_between(trade0.index, sp, -p.threshold, where=sp < -p.threshold, alpha=0.15, color=color)
    ax.set_title(f"Spread {label} — {si}/{sj}  |  β={p.beta:.3f}  σ={p.sigma:.3f}")
    ax.set_ylabel("Spread")
    ax.legend(fontsize=9)

axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.savefig("fig2_spread.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Lancer le pipeline complet en ligne de commande

Pour la réplication complète (415 tickers, 7 périodes, 4 méthodes) :

```bash
# Toutes les méthodes
python pipeline.py --prices sp500_prices.csv --wavelet sym22 --n_pairs 1000

# MD uniquement (sans Johansen, rapide)
python pipeline.py --prices sp500_prices.csv --no_coint --n_pairs 1000
```
